In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

In [2]:
builder = SparkSession.builder \
    .appName("Verify-Gold") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "admin123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [3]:
print("=" * 80)
print("GOLD LAYER VERIFICATION")
print("=" * 80)

# Read Gold tables
customer_360 = spark.read.format("delta").load("s3a://gold/customer_360")
aml_dashboard = spark.read.format("delta").load("s3a://gold/aml_compliance_dashboard")
regulatory_report = spark.read.format("delta").load("s3a://gold/cbuae_monthly_report")
ml_features = spark.read.format("delta").load("s3a://gold/ml_features_churn_prediction")

print(f"\n✅ Customer 360: {customer_360.count():,} records")
print(f"✅ AML Dashboard: {aml_dashboard.count():,} records")
print(f"✅ Regulatory Report: {regulatory_report.count():,} records")
print(f"✅ ML Features: {ml_features.count():,} records")

GOLD LAYER VERIFICATION

✅ Customer 360: 10,000 records
✅ AML Dashboard: 5,000 records
✅ Regulatory Report: 1 records
✅ ML Features: 10,000 records


In [4]:
# Customer 360 insights
print("\n📊 CUSTOMER 360 INSIGHTS")
print("\nEngagement Distribution:")
customer_360.groupBy("engagement_tier").count().orderBy("count", ascending=False).show()

print("\nChurn Risk Distribution:")
customer_360.groupBy("churn_risk").count().orderBy("count", ascending=False).show()

print("\nTop Revenue Customers:")
customer_360.select(
    "customer_id", "first_name", "last_name", 
    "customer_segment", "total_balance_aed", 
    "estimated_annual_revenue_aed"
).orderBy("estimated_annual_revenue_aed", ascending=False).show(5, truncate=False)


📊 CUSTOMER 360 INSIGHTS

Engagement Distribution:
+------------------+-----+
|   engagement_tier|count|
+------------------+-----+
|    HIGHLY_ENGAGED| 4323|
|MODERATELY_ENGAGED| 3927|
|    LOW_ENGAGEMENT| 1750|
+------------------+-----+


Churn Risk Distribution:
+-----------------+-----+
|       churn_risk|count|
+-----------------+-----+
|   LOW_CHURN_RISK| 9477|
|MEDIUM_CHURN_RISK|  523|
+-----------------+-----+


Top Revenue Customers:
+-----------+-----------+----------+----------------+-----------------+----------------------------+
|customer_id|first_name |last_name |customer_segment|total_balance_aed|estimated_annual_revenue_aed|
+-----------+-----------+----------+----------------+-----------------+----------------------------+
|C007688    |PAMELA     |WALSH     |PRIVATE_BANKING |54726368.05000   |1148038.33                  |
|C000998    |AMBER      |WASHINGTON|PRIVATE_BANKING |55831646.17000   |1119717.98                  |
|C000559    |HEATHER    |SMITH     |PRIVATE_BAN

In [5]:
# AML Dashboard insights
print("\n🚨 AML DASHBOARD INSIGHTS")
print("\nAlerts by Severity:")
aml_dashboard.groupBy("alert_severity").count().show()

print("\nSLA Status:")
aml_dashboard.groupBy("sla_status").count().show()

print("\nTop Priority Alerts (for analyst assignment):")
aml_dashboard.select(
    "alert_id", "customer_name", "alert_type", 
    "alert_severity", "composite_risk_score", "priority_rank"
).orderBy("priority_rank").show(10, truncate=False)


🚨 AML DASHBOARD INSIGHTS

Alerts by Severity:
+--------------+-----+
|alert_severity|count|
+--------------+-----+
|          HIGH| 1594|
|        MEDIUM| 1566|
|           LOW| 1534|
|      CRITICAL|  306|
+--------------+-----+


SLA Status:
+----------+-----+
|sla_status|count|
+----------+-----+
|   AT_RISK|  819|
|  ON_TRACK|   86|
| COMPLETED| 2196|
|   OVERDUE| 1899|
+----------+-----+


Top Priority Alerts (for analyst assignment):
+-------------+------------------+-----------+--------------+--------------------+-------------+
|alert_id     |customer_name     |alert_type |alert_severity|composite_risk_score|priority_rank|
+-------------+------------------+-----------+--------------+--------------------+-------------+
|AML63A720AA54|GREGORY CASTRO    |STRUCTURING|CRITICAL      |49                  |1            |
|AMLF75BDD4AB8|LORI WADE         |STRUCTURING|CRITICAL      |52                  |2            |
|AMLCDB85630A8|CARRIE ADAMS      |PEP_MATCH  |CRITICAL      |107      

In [6]:
# ML Features insights
print("\n🤖 ML FEATURE STORE INSIGHTS")
print("\nChurn Target Distribution:")
ml_features.groupBy("churn_target").count().show()

print("\nFeature Statistics:")
ml_features.select("recency_days", "frequency_90d", "monetary_90d").describe().show()

print("\n" + "=" * 80)
print("✅ GOLD LAYER VERIFICATION COMPLETE!")
print("=" * 80)

spark.stop()


🤖 ML FEATURE STORE INSIGHTS

Churn Target Distribution:
+------------+-----+
|churn_target|count|
+------------+-----+
|           0|10000|
+------------+-----+


Feature Statistics:
+-------+------------------+-----------------+-----------------+
|summary|      recency_days|    frequency_90d|     monetary_90d|
+-------+------------------+-----------------+-----------------+
|  count|              9518|            10000|            10000|
|   mean|12.854591300693423|           9.4048| 432749.851337724|
| stddev|14.438132769769755|7.374833949152877|685697.9891798358|
|    min|                 1|                0|          0.00000|
|    max|                90|               54|    6532037.74750|
+-------+------------------+-----------------+-----------------+


✅ GOLD LAYER VERIFICATION COMPLETE!
